# rtp-audit pilot on Colab (H200 141 GB or 96 GB GPU)

**Before running:** add `HF_TOKEN` in Colab *Secrets* (accept the Llama 3.1 licence on the Hub first).

How this notebook is meant to be used:
1. Run the cells top to bottom. Every stage prints live progress (one line per ~10% with ETA), marks warnings ⚠️ and errors ❌, and records them in `runs/<config>/events.jsonl` on Drive.
2. Every stage is resumable: after a disconnect, re-run cell 1, then cell 6 again. Finished shards and finished stages are reused when their code and inputs are unchanged.
3. When something fails or looks wrong, run the **last cell**. It writes a *feedback bundle* (environment, stage status, errors with tracebacks, warnings, key results, suggested next actions). Give that file to the coding agent ([docs/AGENT_LOOP.md](../docs/AGENT_LOOP.md)), pull the fixed code (cell 2), and re-run: only stages whose code or inputs changed are recomputed.

In [ ]:
# Cell 1: mount Drive and set environment variables  (spec §13, cell 1)
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

os.environ["RTP_WORK_DIR"] = "/content/drive/MyDrive/rtp"
os.environ["HF_HOME"] = "/content/hf"          # model cache on local disk, not Drive
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:                        # missing secret: gated Llama repos will fail, others work
    print("⚠️ HF_TOKEN secret not available:", type(exc).__name__)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["RTP_PROGRESS"] = "log"              # readable progress lines instead of carriage-return bars
os.makedirs(os.environ["RTP_WORK_DIR"], exist_ok=True)

CONFIG = "configs/pilot.yaml"
MODELS = [
    "allenai/Llama-3.1-Tulu-3-8B-SFT",
    "allenai/Llama-3.1-Tulu-3-8B-DPO",
]

In [ ]:
# Cell 2: get the code and install  (spec §13, cell 2; torch is never reinstalled)
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "<YOUR_REPO_URL>"  # replace before running
PIN_COMMIT = ""               # optional: a commit/tag to run exactly that code (recommended for long pilots)
!test -d /content/rtp-audit || git clone {REPO_URL} /content/rtp-audit
%cd /content/rtp-audit
!git pull
if PIN_COMMIT:
    !git checkout {PIN_COMMIT}
import torch, sys, importlib
torch_before = torch.__version__
!pip -q install -e ".[dev]"
!git log -1 --oneline
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"
sys.path.insert(0, "/content/rtp-audit"); importlib.invalidate_caches()
print("torch before install:", torch_before, "(must be unchanged above)")

In [ ]:
# Cell 3: preflight. GPU, bf16, library pins, HF token, disks, Drive, data, Hub access to MODELS
from rtp import notebook as nb
nb.run("doctor", "--config", CONFIG)
OVR = nb.gpu_overrides()      # batch sizes for this GPU (never part of a stage's identity: safe to change)
print("batch overrides:", OVR)

In [ ]:
# Cell 4: smoke test and the test suite (a few minutes)  (spec §13, cell 3)
nb.run("smoke")
nb.pytest(CONFIG)             # RTP_REQUIRE_INTEGRATION=1: on Colab the smoke model must load, skips count as failures

In [ ]:
# Cell 5: generate the data (only once; reused afterwards)  (spec §13, cell 4)
nb.run("build-data", "--config", CONFIG, strict=True)
nb.show(f"{os.environ['RTP_WORK_DIR']}/runs/pilot/data/data_report.md", max_chars=4000)

In [ ]:
# Cell 6: behavioural measurement + M3 activations/probes; resumable, re-run after an interruption  (spec §13, cell 5)
for m in MODELS:
    for stage in ("measure", "open-ended", "extract"):
        nb.run(*OVR, stage, "--config", CONFIG, "--model", m)   # a failure is recorded; the loop continues
    nb.run("probe", "--config", CONFIG, "--model", m)
    nb.run("status", "--config", CONFIG)

In [ ]:
# Cell 7: summaries  (spec §13, cell 6)
nb.run("report", "--config", CONFIG)
RUN = f"{os.environ['RTP_WORK_DIR']}/runs/pilot"
nb.show(f"{RUN}/report.md")
for m in MODELS:
    slug = m.replace("/", "__")
    nb.show(f"{RUN}/{slug}/probe/summary.md", max_chars=6000)   # M4 readiness warnings

In [ ]:
# Cell 8: feedback bundle for the coding agent (run this whenever something failed or looks wrong)
bundle = nb.feedback(CONFIG)
if bundle:
    nb.show(bundle, max_chars=8000)
    from google.colab import files
    files.download(str(bundle))   # or open it on Drive: runs/pilot/feedback/